In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [2]:
import jax.numpy as jnp

from hcnn.optimization.solver_osqp import (
    OSQPParameters,
    OSQPProblem,
    osqp_solve,
)

A = jnp.array([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0]])


def matrix_vector_product(x):
    return A @ x


def vector_matrix_product(y):
    return A.T @ y


osqp_problem = OSQPProblem(
    x0=jnp.array([0.0, 0.0]),
    cost_quad_diag=jnp.array([1.0, 1.0]),
    cost_linear=jnp.array([0.0, 0.0]),
    lower_bound=jnp.array([0.5, 0.5, 0.0]),
    upper_bound=jnp.array([1.0, 1.0, 0.6]),
    matrix_vector_product=matrix_vector_product,
    vector_matrix_product=vector_matrix_product,
)

osqp_variables = osqp_solve(
    problem=osqp_problem,
    parameters=OSQPParameters(),
)

print(osqp_variables)

OSQPVariables(x_tilde=Array([0.50003105, 0.49996898], dtype=float32), z_tilde=Array([0.50003105, 0.49996898, 0.50003105], dtype=float32), x=Array([0.49997938, 0.49998382], dtype=float32), z=Array([0.5       , 0.5       , 0.49997938], dtype=float32), y=Array([-0.49994728, -0.49998394,  0.        ], dtype=float32), primal_residuals=Array([-2.0623207e-05, -1.6182661e-05,  0.0000000e+00], dtype=float32), dual_residuals=Array([ 3.2097101e-05, -1.1920929e-07], dtype=float32), rho_residual_mult=Array(8., dtype=float32), iteration=Array(13, dtype=int32, weak_type=True))


In [3]:
import jax
import jax.numpy as jnp
from flax import struct

from hcnn.optimization.solver_sqp import (
    SQPParameters,
    SQPProblemBase,
    sqp_solve,
)


@struct.dataclass
class SimpleProblem(SQPProblemBase[jax.Array]):
    def get_x0(self, phi):
        return jnp.zeros_like(self.cost_quad_diag)

    def constraint_fn_phi(self, x: jax.Array, phi: jax.Array) -> jax.Array:
        return phi * x

    @staticmethod
    def factory():
        return SimpleProblem(
            cost_quad_diag=jnp.array([1.0, 1.0]),
            cost_linear=jnp.array([0, 0]),
            phi=jnp.array([1.0, 1.0]),
            lower_bound=jnp.array([0.5, 0.5]),
            upper_bound=jnp.array([1.0, 1.0]),
        )


osqp_variables = sqp_solve(
    SimpleProblem.factory(),
    SQPParameters(),
)

print(osqp_variables)

[0.4999365 0.4999365]


In [4]:
from functools import partial

import jax
from jax import numpy as jnp
from jax.test_util import check_vjp

phi_test = jnp.array([1.0, 1.0])


# Define a simple test function that uses sqp_solve
def test_function(phi):
    problem = SimpleProblem.factory()
    result = sqp_solve(
        problem.replace(phi=phi),
        SQPParameters(),
    )
    return result


check_vjp(
    f=test_function,
    f_vjp=partial(jax.vjp, test_function),
    args=(phi_test,),
    rtol=5 / 100,
)

In [5]:
@struct.dataclass
class QuadProblem(SQPProblemBase[jax.Array]):
    def get_x0(self, phi):
        return jnp.array([2.0, 2.0])

    def constraint_fn_phi(self, x: jax.Array, phi: jax.Array) -> jax.Array:
        return jnp.concatenate(
            [x[:2], jnp.array([phi[0] * x[0] - phi[1] * x[1] ** 2])]
        )

    @staticmethod
    def factory():
        return QuadProblem(
            cost_quad_diag=jnp.array([0.0, 0.0]),
            cost_linear=jnp.array([0.0, -5]),
            phi=jnp.array([1.0, 1.0]),
            lower_bound=jnp.array([1.0, 1.0, 0.0]),
            upper_bound=jnp.array([2.0, 2.0, 0.0]),
        )


osqp_variables = sqp_solve(
    QuadProblem.factory(),
    SQPParameters(),
)

print(osqp_variables)

[1.9999924 1.4142108]


In [6]:
from functools import partial

import jax
from jax import numpy as jnp
from jax.test_util import check_vjp

phi_test = jnp.array([1.0, 1.0])


# Define a simple test function that uses sqp_solve
def test_function(phi):
    problem = QuadProblem.factory()
    result = sqp_solve(
        problem.replace(phi=phi),
        SQPParameters(),
    )
    return result


check_vjp(
    f=test_function,
    f_vjp=partial(jax.vjp, test_function),
    args=(phi_test,),
)